This section initializes the computational environment by importing the necessary Qiskit Nature and Qiskit Algorithms frameworks. It establishes the bridge between Second Quantization (Fermionic states) and Qubit logic. Key imports include the JordanWignerMapper, which is mathematically responsible for translating the anti-commutation relations of electrons into the Pauli spin-gate operations required by the Quantum Processing Unit (QPU).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from qiskit_algorithms import VQE
from qiskit_algorithms.optimizers import SLSQP
from qiskit_nature.second_q.drivers import PySCFDriver
from qiskit_nature.second_q.mappers import JordanWignerMapper
from qiskit_nature.second_q.circuit.library import UCCSD
# Import the GroundStateEigensolver specifically
from qiskit_nature.second_q.algorithms import GroundStateEigensolver
# We use the standard Estimator which is compatible with the latest Qiskit
from qiskit.primitives import Estimator
from qiskit.primitives import Estimator as QiskitEstimator # V2 Estimator
from qiskit_ibm_runtime import QiskitRuntimeService, Estimator, Session

Here, the local research station authenticates with the IBM Quantum Runtime Service. By defining a specific backend (e.g., ibm_brisbane), we prepare the software to move from local classical simulation to real-world quantum hardware. This connection allows for the execution of hybrid algorithms where the 'Heavy Physics' is calculated on a 127-qubit system while the optimization parameters are managed locally.

In [ ]:
# --- 1. IBM CLOUD AUTHENTICATION ---
service = QiskitRuntimeService(channel="ibm_quantum", token=" ")
backend = service.backend("ibm_marrakesh") # Use a utility-scale 127-qubit system

This is the core of the experiment. The function run_proca_simulation acts as a virtual Proca-Universe. It first builds a molecular proxy (H₂) using the PySCFDriver. Crucially, it then manually overrides the electronic integrals by applying a Yukawa Scaling Factor ($e^{-\mu r}$). This modifies the electron-electron repulsion to reflect a universe where the photon has a non-zero rest mass ($\mu$). It then initializes the UCCSD Ansatz—a parameterized quantum circuit that simulates the electron's 'wrestling' movement through the molecule to find the most stable interference pattern.

In [ ]:
# --- 2. THE PROCA EXPERIMENT FUNCTION ---
def run_proca_simulation(mu_val):
    # Driver for H2 proxy as per Chapter 2 of your thesis
    driver = PySCFDriver(atom="H 0 0 0; H 0 0 0.735", basis="sto3g")
    problem = driver.run()

    # Modify integrals using the Yukawa Factor: e^(-mu * r)[cite: 1, 3]
    proca_factor = np.exp(-mu_val * 0.735) 
    problem.hamiltonian.electronic_integrals.alpha_beta *= proca_factor

    mapper = JordanWignerMapper()
    ansatz = UCCSD(problem.num_spatial_orbitals, problem.num_particles, mapper)
    
    # Standard Estimator for legacy compatibility
    estimator = Estimator() 
    vqe = VQE(estimator, ansatz, SLSQP(maxiter=25))
    calc = GroundStateEigensolver(mapper, vqe)
    
    result = calc.solve(problem)
    return result.total_energies[0]

This iterative loop performs the actual Phenomenology. By sweeping through a range of theoretical photon masses ($\mu$), the code executes multiple VQE (Variational Quantum Eigensolver) cycles. Each iteration finds the ground state energy for a specific 'Proca-Mass' value. This systematic sweep allows us to observe how the 'Phase-Binding' effect of the Yukawa potential changes the total energy of a molecular junction—providing the raw data needed to prove the 'Sub-9nm' quenching theory.

In [ ]:
# --- 3. DATA COLLECTION (PHENOMENOLOGY) ---
mass_range = [0.0, 0.05, 0.1, 0.15] # Testing different 'mu' values
energies = []

print("Starting Multi-Mass Proca Simulation...")
for mu in mass_range:
    energy = run_proca_simulation(mu)
    energies.append(energy)
    print(f"Mu: {mu} | Energy: {energy:.6f} Hartree")

The final cell utilizes Matplotlib to generate a graphical representation of the results. By plotting 'Ground State Energy' against 'Photon Mass,' we can visually identify trends in quantum stability. In the context of the thesis, a rising energy curve indicates that the Proca-Yukawa field is successfully 'localizing' the electron, proving that the phase-interference nodes are becoming more robust—a critical requirement for building Topologically Protected Qubits and ultra-efficient transistors.

In [ ]:
# --- 4. VISUALIZE THE "PHASE BINDING" ---
plt.plot(mass_range, energies, marker='o', color='#003366')
plt.title("Impact of Photon Mass (Proca) on Molecular Ground State")
plt.xlabel("Photon Mass (mu)")
plt.ylabel("Ground State Energy (Hartree)")
plt.grid(True)
plt.show()